# 7. Parameter Sensitivity

This notebook tests how changes to different components of the parameter vector affect convergence.

In [ ]:
import Pkg
Pkg.activate(joinpath(@__DIR__, ".."))
using LinkedMasses

From Notebook 4, the maximum allowed error in the norm of $\zeta$ is around 694.

In [ ]:
# Parameters of the actual system
V_c = [0.5, 0.5]

m0 = 1000.0
m = 250.0
c0 = 500.0
c = 250.0
L = 100.0

params = LinkedMassesParameters(L, m0, m, c0, c)

ε = 0.7

# Path
R = 100.0
ϕ0 = -π / 2
p_origin = [20.0, 100.0]
path_circle(s) = p_origin + R * [cos(ϕ0 + s), sin(ϕ0 + s)]

# Guidance parameters
U = 3.0
Δ = 15.0
los = LOSParameters(U, Δ)

# Controller gains
k_v = 0.5
γ = 5.0

ctrl = AdaptiveLinearizingController(params, los, path_circle, k_v, γ, ε, V_c)

In [ ]:
ζ_true = parameter_vector(params, ε, V_c)
Δζ = 694.0
ζ_plus = Vector{Float64}[]
ζ_minus = Vector{Float64}[]
for i in 1:length(ζ_true)
    ζ_p = copy(ζ_true)
    ζ_m = copy(ζ_true)
    ζ_p[i] += Δζ
    ζ_m[i] -= Δζ
    push!(ζ_plus, ζ_p)
    push!(ζ_minus, ζ_m)
end

In [ ]:
p0_init = [0.0, 0.0]
θ_init = -π / 2
v0_init = [0.0, 0.0]
ω_init = 0.0
s_init = 0.0

T_stop = 300.0
function simulate_ζ(ζ::Vector{Float64})
    init_state = AdaptiveSimulationState(p0_init, θ_init, v0_init, ω_init, s_init, ζ)
    return simulate(ctrl, init_state, T_stop; reltol=1e-6, saveat=1.0)
end

sol_true = simulate_ζ(ζ_true)
sol_plus = [simulate_ζ(ζ_p) for ζ_p in ζ_plus]
sol_minus = [simulate_ζ(ζ_m) for ζ_m in ζ_minus]

In [ ]:
using Plots, LaTeXStrings, ColorSchemes
using CSV, DataFrames

e_x, e_y, v_x, v_y = get_errors(sol_true, ctrl)
df = DataFrame(time=sol_true.t, e_x_true=e_x, e_y_true=e_y, v_x_true=v_x, v_y_true=v_y)

scheme = ColorSchemes.tab10

fig1 = plot(sol_true.t, e_x, label="true", xlabel="Time (s)", ylabel=L"$e_x$ (m)", title="Path-Following Error in x", legend=:topright, color=:black)
fig2 = plot(sol_true.t, e_y, label="true", xlabel="Time (s)", ylabel=L"$e_y$ (m)", title="Path-Following Error in y", legend=:topright, color=:black)
fig3 = plot(sol_true.t, v_x, label="true", xlabel="Time (s)", ylabel=L"$v_x$ (m/s)", title="Velocity Error in x", legend=:topright, color=:black)
fig4 = plot(sol_true.t, v_y, label="true", xlabel="Time (s)", ylabel=L"$v_y$ (m/s)", title="Velocity Error in y", legend=:topright, color=:black)

for i in 1:length(ζ_true)
    e_x_p, e_y_p, v_x_p, v_y_p = get_errors(sol_plus[i], ctrl)
    e_x_m, e_y_m, v_x_m, v_y_m = get_errors(sol_minus[i], ctrl)

    plot!(fig1, sol_true.t, e_x_p, label="+Δζ[$i]", color=scheme.colors[i])
    plot!(fig1, sol_true.t, e_x_m, label="-Δζ[$i]", color=scheme.colors[i], ls=:dash)

    plot!(fig2, sol_true.t, e_y_p, label="+Δζ[$i]", color=scheme.colors[i])
    plot!(fig2, sol_true.t, e_y_m, label="-Δζ[$i]", color=scheme.colors[i], ls=:dash)

    plot!(fig3, sol_true.t, v_x_p, label="+Δζ[$i]", color=scheme.colors[i])
    plot!(fig3, sol_true.t, v_x_m, label="-Δζ[$i]", color=scheme.colors[i], ls=:dash)

    plot!(fig4, sol_true.t, v_y_p, label="+Δζ[$i]", color=scheme.colors[i])
    plot!(fig4, sol_true.t, v_y_m, label="-Δζ[$i]", color=scheme.colors[i], ls=:dash)

    df[!, Symbol("e_x_plus_$i")] = e_x_p
    df[!, Symbol("e_x_minus_$i")] = e_x_m
    df[!, Symbol("e_y_plus_$i")] = e_y_p
    df[!, Symbol("e_y_minus_$i")] = e_y_m
    df[!, Symbol("v_x_plus_$i")] = v_x_p
    df[!, Symbol("v_x_minus_$i")] = v_x_m
    df[!, Symbol("v_y_plus_$i")] = v_y_p
    df[!, Symbol("v_y_minus_$i")] = v_y_m
end

folder = joinpath(@__DIR__, "csv")
mkpath(folder)
CSV.write(joinpath(folder, "parameter_sensitivity.csv"), df)

plot(fig1, fig2, fig3, fig4, layout=(2,2), size=(900,600))

## Convergence Analysis

The plots are too convoluted. Let's try calculating the settling time and overshoot for each simulation.

In [ ]:
using DifferentialEquations
import LinearAlgebra: norm

function convergence_analysis(ζ::Vector{Float64}; threshold::Real=0.05, T_window::Real=3e3, solver=Tsit5(), solver_kwargs...)
    init_state = AdaptiveSimulationState(p0_init, θ_init, v0_init, ω_init, s_init, ζ)
    prob = ODEProblem(closed_loop_ode!, pack_state(init_state), (0.0, T_window), ctrl)
    sol = solve(prob, solver; solver_kwargs...)
    e_x, e_y, v_x, v_y = get_errors(sol, ctrl)
    e_norm = sqrt.(e_x.^2 + e_y.^2)
    v_norm = sqrt.(v_x.^2 + v_y.^2)
    e_norm0 = e_norm[1]
    v_norm0 = v_norm[1]
    e_overshoot = maximum(e_norm) / e_norm0
    v_overshoot = maximum(v_norm) / v_norm0

    e_violated = e_norm .> e_norm0 * threshold
    v_violated = v_norm .> v_norm0 * threshold
    index_last_violation = findlast(e_violated .| v_violated)
    T_convergence = isnothing(index_last_violation) ? 0.0 : sol.t[index_last_violation]

    return T_convergence, e_overshoot, v_overshoot
end

In [ ]:
using CSV, DataFrames
T_nominal, e_overshoot_nominal, v_overshoot_nominal = convergence_analysis(ζ_true)
println("Nominal parameters: T_convergence = $T_nominal, e_overshoot = $e_overshoot_nominal, v_overshoot = $v_overshoot_nominal")
df_conv = DataFrame(index=Int64[], delta_T_plus=Float64[], delta_T_minus=Float64[],
                    e_overshoot_plus=Float64[], e_overshoot_minus=Float64[],
                    v_overshoot_plus=Float64[], v_overshoot_minus=Float64[])
for i in 1:length(ζ_true)
    T_plus, e_overshoot_plus, v_overshoot_plus = convergence_analysis(ζ_plus[i])
    T_minus, e_overshoot_minus, v_overshoot_minus = convergence_analysis(ζ_minus[i])
    delta_T_plus = T_plus - T_nominal
    delta_T_minus = T_minus - T_nominal
    push!(df_conv, (i, delta_T_plus, delta_T_minus,
                    e_overshoot_plus, e_overshoot_minus,
                    v_overshoot_plus, v_overshoot_minus))
end
CSV.write(joinpath(folder, "convergence_analysis.csv"), df_conv)
df_conv

### Convergence of the Simulation from NB 3

In [ ]:
k_err = 2
params_estimate = LinkedMassesParameters(L, m0, k_err*m, c0, c/k_err)
ζ_hat0 = parameter_vector(params_estimate, ε, zeros(2))

convergence_analysis(ζ_hat0)